Smoke test run by `tests/run.sh` in CI. Mirrors the cheatsheet setup cells; asserts fail the build.

In [ ]:
# Postgres via jupysql, as in the cheatsheets
%load_ext sql
%sql postgresql://postgres:example@db

In [ ]:
result = %sql SELECT count(*) AS n FROM customers
assert list(result)[0][0] == 2, result

In [ ]:
import json
from pyspark import SparkContext
from jsoniq import RumbleSession

spark = RumbleSession \
    .builder \
    .appName("Big Data examination") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
%load_ext sparksql_magic

orders_df = spark.read.json('fixtures/orders.jsonl')
orders_df.createOrReplaceTempView("orders")
assert orders_df.count() == 3

In [ ]:
%%sparksql
SELECT first(customer), SUM(flat_items.quantity) AS q
FROM orders LATERAL VIEW EXPLODE(items) AS flat_items
WHERE flat_items.product == "computer screen"
GROUP BY order_id

In [ ]:
rows = spark.sql("""
    SELECT SUM(i.quantity) FROM orders LATERAL VIEW EXPLODE(items) AS i
    WHERE i.product = 'computer screen'""").collect()
assert rows[0][0] == 12, rows

In [ ]:
%load_ext jsoniqmagic
conf = spark.getRumbleConf()
conf.set("runtime.resultsSizeCap", 20)
conf.set("runtime.materializationCap", 100000)

In [ ]:
%%jsoniq
json-file("fixtures/events.json", 8)[1].type

In [ ]:
assert spark.jsoniq('json-file("fixtures/events.json", 8)[1].type').json() == ("PushEvent",)
assert spark.jsoniq('count(json-lines("fixtures/orders.jsonl")[$$.customer.first_name eq "Preston"])').json() == (2,)
print("all checks passed")